# A8: RAG system for DataPact

INFO 490 MG, Team 15.

RAG means we look up the right text first, then give it to the model with the question. The steps are:

Data → Embeddings → Retrieval → LLM → Output

We test 3 embedding models with 3 ways of chunking, so 9 setups in total. The generation model
never changes. We use SmolLM2-1.7B-Instruct every time because it was our pick in A6 and our
Django app already runs it.

**How to run:** the steps are in `README.md`. Set `RUN_MODE` in the next cell first.

| RUN_MODE | What it does |
| --- | --- |
| `full` | Real models and all the queries. This is the real run. |
| `quick` | Real models and 2 queries. Run this first to make sure the GPU works. |
| `dry` | Fake models with no downloads. This only checks that the code runs, so ignore its numbers. |

In [1]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_MODE = os.environ.get("RAG_RUN_MODE", "full")  # full, quick or dry

BASE = Path("rag_system") if Path("rag_system").is_dir() else Path(".")

# Ashok's documents go in knowledge_base/. Until then the small sample set is used.
USING_SAMPLE = not any((BASE / "knowledge_base").glob("*.md"))
KB_DIR = BASE / ("sample_kb" if USING_SAMPLE else "knowledge_base")
QUERY_FILE = BASE / ("sample_queries.json" if USING_SAMPLE else "test_queries.json")

# Only a full run on the real documents writes to results/. Anything else stays out of the way.
if RUN_MODE == "full" and not USING_SAMPLE:
    OUT_DIR = BASE / "results"
else:
    OUT_DIR = BASE / f"results_{RUN_MODE}{'_sample' if USING_SAMPLE else ''}"
OUT_DIR.mkdir(exist_ok=True)

print("Run mode:", RUN_MODE)
print("Knowledge base:", KB_DIR)
print("Saving to:", OUT_DIR)
if USING_SAMPLE:
    print("NOTE: using the sample documents. These results are not for the report.")

Run mode: full
Knowledge base: knowledge_base
Saving to: results


## Part 1: The RAG pipeline

### Steps 1.1 and 1.2: Knowledge base

Every `.md` file in the knowledge base is one document.

* `# ` is the title of the document.
* `## ` starts a new section. Each section is about one topic.
* **A paragraph** is a block of text with a blank line before and after it.

In [2]:
def load_paragraphs(kb_dir):
    paragraphs = []
    for path in sorted(Path(kb_dir).glob("*.md")):
        title = path.stem
        section = "General"
        for block in re.split(r"\n\s*\n", path.read_text(encoding="utf-8")):
            block = block.strip()
            if not block:
                continue
            if block.startswith("# "):
                title = block[2:].strip()
            elif block.startswith("## "):
                section = block[3:].strip()
            else:
                text = " ".join(block.split())
                paragraphs.append({"doc": path.name, "title": title, "section": section,
                                   "text": text, "words": len(text.split())})
    return paragraphs


paragraphs = load_paragraphs(KB_DIR)
word_counts = [p["words"] for p in paragraphs]
kb_table = pd.DataFrame(paragraphs)

print(f"Documents: {kb_table['doc'].nunique()}")
print(f"Sections: {kb_table[['doc', 'section']].drop_duplicates().shape[0]}")
print(f"Paragraphs: {len(paragraphs)} (minimum 20, recommended 30 to 50)")
print(f"Total words: {sum(word_counts)} (minimum 2,000)")
print(f"Paragraph words: min {min(word_counts)}, max {max(word_counts)} (target 80 to 200)")

outside = kb_table[(kb_table["words"] < 80) | (kb_table["words"] > 200)]
print(f"Paragraphs outside 80 to 200 words: {len(outside)}")
if not USING_SAMPLE and (len(paragraphs) < 20 or sum(word_counts) < 2000):
    print("WARNING: the knowledge base is below the assignment minimum.")

Documents: 9
Sections: 40
Paragraphs: 40 (minimum 20, recommended 30 to 50)
Total words: 4019 (minimum 2,000)
Paragraph words: min 91, max 115 (target 80 to 200)
Paragraphs outside 80 to 200 words: 0


### Step 1.2: Three ways to chunk

All three follow the rules from the assignment. We never cut a sentence in half, and a chunk never
mixes two documents. Each document is about one topic, like contracts or violations, so text from
the same document is never "unrelated".

In our knowledge base every section is one paragraph, so a **paragraph** is the text under one
`##` heading.

1. **Fixed length:** adds whole sentences to a chunk until it reaches about 120 words, then starts
   the next chunk. It ignores paragraph and section breaks, so one chunk can end partway into the
   next section.
2. **Overlapping paragraphs:** paragraph 1 + 2, then 2 + 3, and so on inside a document. Each chunk
   shares one full paragraph with the next chunk. That shared paragraph is the overlap.
3. **Hybrid:** keeps each section as its own chunk. A very short paragraph gets joined to the one
   next to it, and a very long one gets split at the end of a sentence. We also add the document
   title and section name to the text before embedding it, so the chunk still says what it is
   about. The model only sees the normal text.

A retrieved chunk counts as the right one if it contains the expected section, even if it also
covers the section next to it.

In [3]:
FIXED_WORDS = 120
HYBRID_MIN_WORDS = 60
HYBRID_MAX_WORDS = 200
HYBRID_SPLIT_WORDS = 150


def split_sentences(text):
    protected = re.sub(r"\b(e\.g|i\.e|etc|vs)\.", lambda m: m.group(0).replace(".", "<dot>"), text)
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", protected)
    return [p.replace("<dot>", ".").strip() for p in parts if p.strip()]


def group_by_section(paragraphs):
    sections = {}
    for p in paragraphs:
        sections.setdefault((p["doc"], p["title"], p["section"]), []).append(p["text"])
    return sections


def group_by_doc(paragraphs):
    docs = {}
    for p in paragraphs:
        docs.setdefault(p["doc"], []).append(p)
    return docs


def pack_sentences(sentences, limit):
    groups, current, count = [], [], 0
    for sentence in sentences:
        size = len(sentence.split())
        if current and count + size > limit:
            groups.append(" ".join(current))
            current, count = [], 0
        current.append(sentence)
        count += size
    if current:
        groups.append(" ".join(current))
    return groups


def make_chunk(doc, sections, text, embed_text=None):
    sections = list(dict.fromkeys(sections))
    return {"doc": doc, "sections": sections, "section": " + ".join(sections), "text": text,
            "embed_text": embed_text or text, "words": len(text.split())}


def fixed_chunks(paragraphs):
    chunks = []
    for doc, paras in group_by_doc(paragraphs).items():
        current, sections, count = [], [], 0
        for p in paras:
            for sentence in split_sentences(p["text"]):
                size = len(sentence.split())
                if current and count + size > FIXED_WORDS:
                    chunks.append(make_chunk(doc, sections, " ".join(current)))
                    current, sections, count = [], [], 0
                current.append(sentence)
                sections.append(p["section"])
                count += size
        if current:
            chunks.append(make_chunk(doc, sections, " ".join(current)))
    return chunks


def overlap_chunks(paragraphs):
    chunks = []
    for doc, paras in group_by_doc(paragraphs).items():
        if len(paras) == 1:
            chunks.append(make_chunk(doc, [paras[0]["section"]], paras[0]["text"]))
        for first, second in zip(paras, paras[1:]):
            chunks.append(make_chunk(doc, [first["section"], second["section"]],
                                     first["text"] + " " + second["text"]))
    return chunks


def hybrid_chunks(paragraphs):
    chunks = []
    for (doc, title, section), texts in group_by_section(paragraphs).items():
        merged = []
        for text in texts:
            if merged and len(merged[-1].split()) < HYBRID_MIN_WORDS:
                merged[-1] += " " + text
            else:
                merged.append(text)
        if len(merged) > 1 and len(merged[-1].split()) < HYBRID_MIN_WORDS:
            last = merged.pop()
            merged[-1] += " " + last

        for text in merged:
            pieces = [text]
            if len(text.split()) > HYBRID_MAX_WORDS:
                pieces = pack_sentences(split_sentences(text), HYBRID_SPLIT_WORDS)
            for piece in pieces:
                chunks.append(make_chunk(doc, [section], piece, f"{title} - {section}: {piece}"))
    return chunks


CHUNKERS = {"fixed": fixed_chunks, "overlap": overlap_chunks, "hybrid": hybrid_chunks}
chunk_sets = {name: chunker(paragraphs) for name, chunker in CHUNKERS.items()}

pd.DataFrame([
    {"strategy": name, "chunks": len(chunks),
     "avg_words": round(np.mean([c["words"] for c in chunks]), 1),
     "min_words": min(c["words"] for c in chunks),
     "max_words": max(c["words"] for c in chunks)}
    for name, chunks in chunk_sets.items()
])

,strategy,chunks,avg_words,min_words,max_words
0,fixed,40,100.5,46,119
1,overlap,31,200.0,185,221
2,hybrid,40,100.5,91,115


### Step 1.3: Embeddings

An embedding turns text into a list of numbers. Text with a similar meaning gets similar numbers.
We use three free sentence-transformers models of different sizes:

| Size | Model | Dimensions |
| --- | --- | --- |
| Small | all-MiniLM-L6-v2 | 384 |
| Medium | all-mpnet-base-v2 | 768 |
| Large | BAAI/bge-large-en-v1.5 | 1024 |

The bge model says to add a short line in front of each query, so `query_prefix` does that.

`VectorIndex` keeps all the embeddings in one NumPy matrix. The vectors are normalized, so a dot
product gives us cosine similarity. Our knowledge base is small, so we don't need a vector
database. One matrix multiplication searches everything.

In [4]:
EMBEDDING_MODELS = {
    "small": {"id": "sentence-transformers/all-MiniLM-L6-v2", "dim": 384, "query_prefix": ""},
    "medium": {"id": "sentence-transformers/all-mpnet-base-v2", "dim": 768, "query_prefix": ""},
    "large": {"id": "BAAI/bge-large-en-v1.5", "dim": 1024,
              "query_prefix": "Represent this sentence for searching relevant passages: "},
}
GENERATION_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
MAX_NEW_TOKENS = 160
TOP_K = 3


class RealEmbedder:
    def __init__(self, config, device):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(config["id"], device=device)
        self.query_prefix = config["query_prefix"]

    def encode_chunks(self, texts):
        return self.model.encode(texts, normalize_embeddings=True, batch_size=32,
                                 show_progress_bar=False)

    def encode_query(self, query):
        return self.model.encode([self.query_prefix + query], normalize_embeddings=True,
                                 show_progress_bar=False)[0]


class FakeEmbedder:
    # Dry run only: hashes words into a vector so the code path runs without a download.
    def __init__(self, config, device=None):
        self.dim = config["dim"]

    def _vector(self, text):
        vector = np.zeros(self.dim)
        for word in re.findall(r"[a-z]+", text.lower()):
            vector[sum(ord(c) * 31 ** i for i, c in enumerate(word)) % self.dim] += 1
        return vector / (np.linalg.norm(vector) or 1)

    def encode_chunks(self, texts):
        return np.array([self._vector(t) for t in texts])

    def encode_query(self, query):
        return self._vector(query)


class VectorIndex:
    def __init__(self, embedder, chunks):
        self.embedder = embedder
        self.chunks = chunks
        started = time.perf_counter()
        self.matrix = np.asarray(embedder.encode_chunks([c["embed_text"] for c in chunks]))
        self.build_seconds = time.perf_counter() - started

    def search(self, query, k):
        started = time.perf_counter()
        query_vector = np.asarray(self.embedder.encode_query(query))
        scores = self.matrix @ query_vector
        top = np.argsort(-scores)[:k]
        search_ms = (time.perf_counter() - started) * 1000
        hits = [{**self.chunks[i], "score": float(scores[i]), "rank": rank}
                for rank, i in enumerate(top, start=1)]
        return hits, search_ms

### Step 1.4: Retrieval

This runs one query so you can see each step: turn the query into an embedding, compare it to every
chunk, and keep the top k. The device setup is the same as A6 and A7. Our 7900 XTX uses AMD's ROCm
version of PyTorch, and it still shows up as `cuda`.

In [5]:
if RUN_MODE == "dry":
    DEVICE = "cpu"
else:
    import torch

    if torch.cuda.is_available():
        DEVICE = "cuda"
        print("GPU:", torch.cuda.get_device_name(0))
    elif torch.backends.mps.is_available():
        DEVICE = "mps"
    else:
        DEVICE = "cpu"
print("Device:", DEVICE)

embedders = {}
for size, config in EMBEDDING_MODELS.items():
    embedders[size] = (FakeEmbedder if RUN_MODE == "dry" else RealEmbedder)(config, DEVICE)
    dim = len(embedders[size].encode_query("test"))
    if dim != config["dim"]:
        raise ValueError(f"{config['id']} gave {dim} dimensions, expected {config['dim']}")
    print(f"{size}: {config['id']} ({dim} dimensions)")

queries = json.loads(QUERY_FILE.read_text(encoding="utf-8"))
if RUN_MODE == "quick":
    queries = queries[:2]

example = VectorIndex(embedders["small"], chunk_sets["hybrid"])
hits, search_ms = example.search(queries[0]["query"], TOP_K)
print("\nQuery:", queries[0]["query"], f"({search_ms:.1f} ms)")
for hit in hits:
    print(f"  {hit['rank']}. score {hit['score']:.3f} | {hit['doc']} > {hit['section']}")

GPU: AMD Radeon RX 7900 XTX
Device: cuda


C:\Users\Connor\Documents\UIUC\Fall 2026\INFO 490\llm-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 51528.31it/s]

C:\Users\Connor\Documents\UIUC\Fall 2026\INFO 490\llm-venv\Lib\site-packages\transformers\integrations\sdpa_attention.py:170: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


small: sentence-transformers/all-MiniLM-L6-v2 (384 dimensions)


... (2 outputs left out)

medium: sentence-transformers/all-mpnet-base-v2 (768 dimensions)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5076.34it/s]

large: BAAI/bge-large-en-v1.5 (1024 dimensions)

Query: What does a not null rule check for? (4.8 ms)
  1. score 0.673 | rule_types.md > Not null
  2. score 0.438 | contracts.md > Writing a new rule
  3. score 0.423 | rule_types.md > Allowed values


### Step 1.5: Generation

The prompt has the three parts the assignment asks for: a system instruction, the retrieved chunks
and the question. Loading and generating work the same way as `data_quality/ai/local_model.py`:
16-bit on the GPU, no random sampling, and the model's own chat template.

In [6]:
SYSTEM_PROMPT = (
    "You answer questions about DataPact, a data quality tool. "
    "Use only the context below. "
    "If the context does not contain the answer, say: I could not find that in the documents."
)


def build_messages(query, hits, system_prompt=SYSTEM_PROMPT):
    context = "\n\n".join(f"[{hit['rank']}] {hit['text']}" for hit in hits)
    return [{"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]


class RealGenerator:
    def __init__(self, model_id, device):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        kwargs = {"dtype": torch.float16 if device != "cpu" else torch.float32}
        if device != "cpu":
            kwargs["device_map"] = device
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        self.model.eval()

    def generate(self, messages):
        inputs = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(self.model.device)
        with self.torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                         pad_token_id=self.tokenizer.pad_token_id)
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


class FakeGenerator:
    # Dry run only: repeats the first sentence of the top chunk.
    def generate(self, messages):
        context = messages[1]["content"].split("\n\nQuestion:")[0]
        return split_sentences(context.split("] ", 1)[1])[0]


generator = FakeGenerator() if RUN_MODE == "dry" else RealGenerator(GENERATION_MODEL, DEVICE)

# The first call is slow while the GPU warms up, so it is not timed in the experiments.
print(generator.generate(build_messages(queries[0]["query"], hits)))

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading weights:   8%|▊         | 18/218 [00:00<00:01, 140.06it/s]

Loading weights:  15%|█▌        | 33/218 [00:00<00:01, 143.76it/s]

Loading weights:  22%|██▏       | 48/218 [00:00<00:01, 137.32it/s]

Loading weights:  28%|██▊       | 62/218 [00:00<00:01, 125.09it/s]

Loading weights:  36%|███▌      | 79/218 [00:00<00:00, 139.45it/s]

... (6 outputs left out)

Loading weights:  85%|████████▌ | 186/218 [00:01<00:00, 129.22it/s]

Loading weights:  92%|█████████▏| 201/218 [00:01<00:00, 133.55it/s]

Loading weights: 100%|██████████| 218/218 [00:01<00:00, 136.83it/s]

A not-null rule asserts that a column always carries a value. It fails on empty strings and on missing fields as well as on true nulls, because all three mean the same thing to somebody reading a report. This rule needs no parameters. It is the cheapest check to write and usually the first one worth adding to a new contract, since a column that is sometimes blank will break almost any downstream calculation. When it fails on a large share of rows, the usual cause is an upstream export that changed which columns it includes rather than a genuine data-entry problem, so check the file header before chasing individual records.


## Part 2: Experiments

### Step 2.1: Test queries

The queries are in `test_queries.json`. Each one says where the answer is and which phrases a good
answer should have, so every setup gets graded the same way. If a query has `"out_of_scope": true`,
the answer is not in our documents, so the right move is for the model to say that.

In [7]:
pd.DataFrame(queries)

,id,query,expected_doc,expected_section,expected_keywords,difficulty,note,out_of_scope
0,q01,What does a not null rule check for?,rule_types.md,Not null,"[empty strings, no parameters]",easy,Direct keyword overlap with one section.,NaN
1,q02,How do I register a new dataset in DataPact?,getting_started.md,Registering a dataset,"[Manage Datasets, unique for the owner]",easy,Direct keyword overlap.,NaN
2,q03,What are the three statuses a contract version...,contracts.md,Activating and retiring a version,"[draft, active, retired]",easy,Enumeration lives in one section.,NaN
3,q04,Can I change a contract after it has already b...,contracts.md,Editing a contract that has been used,"[new version, never be edited]",medium,Paraphrased; competes with 'Contract versions'.,NaN
4,q05,Why won't the system let me delete a teammate'...,getting_started.md,Who owns a dataset,"[reassigned, accountable]",medium,No shared vocabulary with the target text.,NaN
5,q06,What is the difference between a run that fail...,validation_runs.md,Run outcomes,"[could not be evaluated, never happened]",medium,Competes with troubleshooting.md on the same d...,NaN
6,q07,Duplicate IDs showed up in a key column. How d...,rule_types.md,Unique,"[exported twice, failed rows against the total]",medium,Diagnostic question; answer is one clause in a...,NaN
7,q08,One new code appeared in thousands of rows. Is...,violations.md,Data issue or contract updated,"[new legitimate category, How many rows failed]",hard,Three sections discuss this; only one gives th...,NaN
8,q09,If a program is going to read the download rat...,reports_and_exports.md,JSON export,"[read by a program, how many records]",hard,Directly competes with the CSV export section.,NaN
9,q10,Can anyone see our data without signing in?,accounts_and_access.md,Protected and public endpoints,"[run-outcome summary, unauthorised]",hard,Competes with 'What anonymous visitors can see'.,NaN


### Step 2.2: Evaluation table

Every setup answers every query. The `_auto` columns are a first guess based on simple rules:

* **context_quality_auto:** Good means we found the right section and at least 75% of the expected
  phrases. Partial means we only got one of those, or at least half the phrases. Anything else is
  Poor.
* **answer_quality_auto:** same idea, but it checks the answer. Good is 75% of the phrases or more,
  and Partial is 40% or more. For an out-of-scope query, Good means the model said it didn't know
  and Poor means it made something up.

Matching phrases is rough. It can mark a right answer as wrong, or a wrong answer as right. So
`context_quality_manual`, `answer_quality_manual` and `notes` are left blank for us to fill in after
reading the answers. The full retrieved chunks are saved too.

In [8]:
REFUSAL = re.compile(r"could not find|cannot find|can't find|not in the (context|documents)"
                     r"|(does|do) not (contain|say|mention|provide|specify)"
                     r"|not (mentioned|specified|provided|stated|covered|included)|no information", re.I)


def phrase_recall(text, phrases):
    if not phrases:
        return float("nan")
    text = text.lower()
    return sum(p.lower() in text for p in phrases) / len(phrases)


def is_gold(hit, query):
    return hit["doc"] == query.get("expected_doc") and query.get("expected_section") in hit["sections"]


def score_retrieval(query, hits):
    if query.get("out_of_scope"):
        return {"context_hit": None, "first_hit_rank": None, "context_phrase_recall": None,
                "context_quality_auto": "n/a"}
    ranks = [hit["rank"] for hit in hits if is_gold(hit, query)]
    recall = phrase_recall(" ".join(hit["text"] for hit in hits), query.get("expected_keywords", []))
    hit = bool(ranks)
    if hit and recall >= 0.75:
        label = "Good"
    elif hit or recall >= 0.5:
        label = "Partial"
    else:
        label = "Poor"
    return {"context_hit": int(hit), "first_hit_rank": min(ranks) if ranks else None,
            "context_phrase_recall": round(recall, 2), "context_quality_auto": label}


def score_answer(query, answer):
    declined = bool(REFUSAL.search(answer))
    if query.get("out_of_scope"):
        return {"answer_phrase_recall": None, "declined": declined,
                "answer_quality_auto": "Good" if declined else "Poor"}
    recall = phrase_recall(answer, query.get("expected_keywords", []))
    label = "Good" if recall >= 0.75 else "Partial" if recall >= 0.4 else "Poor"
    return {"answer_phrase_recall": round(recall, 2), "declined": declined,
            "answer_quality_auto": label}


def run_query(index, query, k=TOP_K, system_prompt=SYSTEM_PROMPT):
    hits, search_ms = index.search(query["query"], k)
    started = time.perf_counter()
    answer = generator.generate(build_messages(query["query"], hits, system_prompt))
    generate_seconds = time.perf_counter() - started
    row = {"query_id": query["id"], "query": query["query"], **score_retrieval(query, hits),
           "answer": answer, **score_answer(query, answer),
           "search_ms": round(search_ms, 1), "generate_s": round(generate_seconds, 2),
           "total_s": round(search_ms / 1000 + generate_seconds, 2)}
    return row, hits


indexes = {}
rows, chunk_rows, index_rows = [], [], []
for size in EMBEDDING_MODELS:
    for strategy, chunks in chunk_sets.items():
        index = VectorIndex(embedders[size], chunks)
        indexes[(size, strategy)] = index
        index_rows.append({"embedding_model": size, "chunking_strategy": strategy, "chunks": len(chunks),
                           "dimensions": index.matrix.shape[1],
                           "vectors_mb": round(index.matrix.nbytes / 1e6, 3),
                           "build_seconds": round(index.build_seconds, 3)})
        for query in queries:
            row, hits = run_query(index, query)
            rows.append({"embedding_model": size, "chunking_strategy": strategy, **row,
                         "retrieved_chunks": "\n---\n".join(
                             f"[{h['rank']}] ({h['score']:.3f}) {h['doc']} > {h['section']}: {h['text']}"
                             for h in hits),
                         "context_quality_manual": "", "answer_quality_manual": "", "notes": ""})
            chunk_rows += [{"embedding_model": size, "chunking_strategy": strategy,
                            "query_id": query["id"], "rank": h["rank"], "score": round(h["score"], 4),
                            "doc": h["doc"], "section": h["section"], "is_expected": is_gold(h, query),
                            "chunk_text": h["text"]} for h in hits]
        print(f"done: {size} + {strategy} ({index.build_seconds:.2f}s to embed {len(chunks)} chunks)")

evaluation = pd.DataFrame(rows)
evaluation.to_csv(OUT_DIR / "evaluation_table.csv", index=False)
pd.DataFrame(chunk_rows).to_csv(OUT_DIR / "retrieved_chunks.csv", index=False)
pd.DataFrame(index_rows).to_csv(OUT_DIR / "index_stats.csv", index=False)
evaluation[["embedding_model", "chunking_strategy", "query_id", "context_quality_auto",
            "answer_quality_auto", "total_s"]].head(15)

done: small + fixed (0.06s to embed 40 chunks)


done: small + overlap (0.07s to embed 31 chunks)


done: small + hybrid (0.06s to embed 40 chunks)


done: medium + fixed (0.26s to embed 40 chunks)


done: medium + overlap (0.38s to embed 31 chunks)


done: medium + hybrid (0.26s to embed 40 chunks)


done: large + fixed (0.93s to embed 40 chunks)


done: large + overlap (1.46s to embed 31 chunks)


done: large + hybrid (1.01s to embed 40 chunks)


,embedding_model,chunking_strategy,query_id,context_quality_auto,answer_quality_auto,total_s
0,small,fixed,q01,Good,Partial,0.69
1,small,fixed,q02,Good,Partial,1.55
2,small,fixed,q03,Partial,Poor,0.28
3,small,fixed,q04,Partial,Poor,0.90
4,small,fixed,q05,Good,Partial,2.13
5,small,fixed,q06,Good,Partial,1.03
6,small,fixed,q07,Good,Poor,0.19
7,small,fixed,q08,Good,Poor,0.18
8,small,fixed,q09,Good,Partial,1.40
9,small,fixed,q10,Good,Partial,0.97


### Steps 2.3 and 2.4: Comparing the models and chunking

These tables show averages for each setup, each embedding model and each chunking strategy.
Out-of-scope queries only count toward `declined_correctly`.

In [9]:
def summarize(table, group):
    in_scope = table[table["context_quality_auto"] != "n/a"]
    out_of_scope = table[table["context_quality_auto"] == "n/a"]
    summary = in_scope.groupby(group).agg(
        context_hit_rate=("context_hit", "mean"),
        context_phrase_recall=("context_phrase_recall", "mean"),
        answer_phrase_recall=("answer_phrase_recall", "mean"),
        avg_search_ms=("search_ms", "mean"),
        avg_generate_s=("generate_s", "mean"),
    ).round(3)
    if len(out_of_scope):
        summary["declined_correctly"] = out_of_scope.groupby(group)["declined"].mean().round(3)
    return summary


by_config = summarize(evaluation, ["embedding_model", "chunking_strategy"])
by_model = summarize(evaluation, "embedding_model")
by_strategy = summarize(evaluation, "chunking_strategy")

by_config.to_csv(OUT_DIR / "summary_by_config.csv")
by_model.to_csv(OUT_DIR / "summary_by_model.csv")
by_strategy.to_csv(OUT_DIR / "summary_by_strategy.csv")

ranked = by_config.sort_values(["context_hit_rate", "answer_phrase_recall", "avg_search_ms"],
                               ascending=[False, False, True])
BEST = ranked.index[0]
print("Best configuration:", BEST)
display(by_model)
display(by_strategy)
ranked

Best configuration: ('medium', 'fixed')


,context_hit_rate,context_phrase_recall,answer_phrase_recall,avg_search_ms,avg_generate_s,declined_correctly
embedding_model,,,,,,
large,0.889,0.819,0.375,16.017,1.069,1.0
medium,0.972,0.958,0.408,9.078,1.194,1.0
small,0.944,0.889,0.347,5.131,1.208,1.0


,context_hit_rate,context_phrase_recall,answer_phrase_recall,avg_search_ms,avg_generate_s,declined_correctly
chunking_strategy,,,,,,
fixed,1.000,0.861,0.352,10.239,0.909,1.0
hybrid,0.944,0.944,0.333,9.767,1.062,1.0
overlap,0.861,0.861,0.444,10.219,1.500,1.0


context_hit_rate  context_phrase_recall  \
embedding_model chunking_strategy                                            
medium          fixed                         1.000                  0.958   
small           hybrid                        1.000                  1.000   
large           fixed                         1.000                  0.792   
small           fixed                         1.000                  0.833   
medium          hybrid                        1.000                  1.000   
                overlap                       0.917                  0.917   
small           overlap                       0.833                  0.833   
large           overlap                       0.833                  0.833   
                hybrid                        0.833                  0.833   

                                   answer_phrase_recall  avg_search_ms  \
embedding_model chunking_strategy                                        
medium          fixed                             0.472          9.825   
small           hybrid                            0.375          5.508   
large           fixed                             0.333         16.017   
small           fixed                             0.250          4.875   
medium          hybrid                            0.250          7.983   
                overlap                           0.500          9.425   
small           overlap                           0.417          5.008   
large           overlap                           0.417         16.225   
                hybrid                            0.375         15.808   

                                   avg_generate_s  declined_correctly  
embedding_model chunking_strategy                                      
medium          fixed                       1.056                 1.0  
small           hybrid                      1.273                 1.0  
large           fixed                       0.852                 1.0  
small           fixed                       0.821                 1.0  
medium          hybrid                      0.898                 1.0  
                overlap                     1.628                 1.0  
small           overlap                     1.530                 1.0  
large           overlap                     1.342                 1.0  
                hybrid                      1.014                 1.0

### Similarity threshold

A threshold lets the app say "nothing found" without asking the model, when even the best chunk
is not similar enough. For each setup we take the top score of every query. A good threshold sits
between the lowest score of a question we can answer and the highest score of a question we
can't.

* **gap** above zero means one cutoff separates them, and the suggested threshold is the middle.
* **gap** at zero or below means no cutoff works without blocking a real question. Then the prompt
  has to handle refusals.

Scores mean different things for each embedding model, so a threshold only works for the model it
came from. We only have 2 out-of-scope queries, so treat it as a starting point.

In [10]:
threshold_rows = []
for (size, strategy), index in indexes.items():
    for query in queries:
        hits, _ = index.search(query["query"], 1)
        threshold_rows.append({"embedding_model": size, "chunking_strategy": strategy,
                               "query_id": query["id"], "out_of_scope": bool(query.get("out_of_scope")),
                               "top_score": round(hits[0]["score"], 4)})
threshold_scores = pd.DataFrame(threshold_rows)
threshold_scores.to_csv(OUT_DIR / "threshold_scores.csv", index=False)

summary_rows = []
for (size, strategy), group in threshold_scores.groupby(["embedding_model", "chunking_strategy"]):
    answerable = group.loc[~group["out_of_scope"], "top_score"]
    unanswerable = group.loc[group["out_of_scope"], "top_score"]
    row = {"embedding_model": size, "chunking_strategy": strategy,
           "lowest_answerable": answerable.min(), "highest_unanswerable": None,
           "gap": None, "suggested_threshold": None}
    if len(unanswerable):
        gap = answerable.min() - unanswerable.max()
        row.update(highest_unanswerable=unanswerable.max(), gap=round(gap, 4))
        if gap > 0:
            row["suggested_threshold"] = round((answerable.min() + unanswerable.max()) / 2, 4)
    summary_rows.append(row)

threshold_summary = pd.DataFrame(summary_rows)
threshold_summary.to_csv(OUT_DIR / "threshold_summary.csv", index=False)
best_row = threshold_summary[(threshold_summary["embedding_model"] == BEST[0])
                             & (threshold_summary["chunking_strategy"] == BEST[1])].iloc[0]
print("Best setup:", BEST, "| suggested threshold:", best_row["suggested_threshold"])
threshold_summary

Best setup: ('medium', 'fixed') | suggested threshold: nan


,embedding_model,chunking_strategy,lowest_answerable,highest_unanswerable,gap,suggested_threshold
0,large,fixed,0.5900,0.5816,0.0084,0.5858
1,large,hybrid,0.5486,0.6061,-0.0575,NaN
2,large,overlap,0.5985,0.5913,0.0072,0.5949
3,medium,fixed,0.3562,0.4169,-0.0607,NaN
4,medium,hybrid,0.3459,0.4294,-0.0835,NaN
5,medium,overlap,0.2891,0.4056,-0.1165,NaN
6,small,fixed,0.4155,0.4568,-0.0413,NaN
7,small,hybrid,0.3648,0.4440,-0.0792,NaN
8,small,overlap,0.3697,0.3681,0.0016,0.3689


### Step 2.5: Data scaling

This uses the best setup from above. The chunks with the answer stay in every test, and we keep
25%, 50% and then 100% of the other chunks. The answer is always there, so any change comes from
the extra text getting in the way.

* **hit:** how often the right section is in the top k.
* **noise:** how much of what we retrieved came from the wrong section.
* **margin:** the best right chunk's score minus the best wrong chunk's score. A smaller margin
  means the wrong text is getting closer. Below zero means the wrong text won.

In [11]:
SCALE_LEVELS = [0.25, 0.5, 1.0]
in_scope_queries = [q for q in queries if not q.get("out_of_scope")]
best_size, best_strategy = BEST
all_chunks = chunk_sets[best_strategy]
rng = np.random.default_rng(42)
shuffled = [all_chunks[i] for i in rng.permutation(len(all_chunks))]

scale_rows = []
for level in SCALE_LEVELS:
    keep = shuffled[:max(1, round(len(shuffled) * level))]
    for query in in_scope_queries:
        gold = [c for c in all_chunks if is_gold(c, query)]
        chunks = gold + [c for c in keep if not is_gold(c, query)]
        index = VectorIndex(embedders[best_size], chunks)
        hits, _ = index.search(query["query"], TOP_K)
        all_hits, _ = index.search(query["query"], len(chunks))
        gold_scores = [h["score"] for h in all_hits if is_gold(h, query)]
        other_scores = [h["score"] for h in all_hits if not is_gold(h, query)]
        margin = max(gold_scores) - max(other_scores) if gold_scores and other_scores else None
        scale_rows.append({"share_of_data": level, "chunks": len(chunks), "query_id": query["id"],
                           "hit": int(any(is_gold(h, query) for h in hits)),
                           "noise": 1 - sum(is_gold(h, query) for h in hits) / len(hits),
                           "margin": margin})

scaling = pd.DataFrame(scale_rows)
scaling.to_csv(OUT_DIR / "scaling_results.csv", index=False)
scaling.groupby("share_of_data")[["chunks", "hit", "noise", "margin"]].mean().round(3)

,chunks,hit,noise,margin
share_of_data,,,,
0.25,11.333,1.0,0.500,0.223
0.50,20.833,1.0,0.500,0.186
1.00,40.000,1.0,0.556,0.128


## Part 3: Failure analysis and improvement

### Step 3.1: Failure cases

These are the worst rows from the evaluation table, with their chunks and answers. Read them and
pick at least 3 real failures for `rag_analysis.md`. Some may turn out fine, since the auto score
can be wrong.

In [12]:
RANK = {"Poor": 0, "Partial": 1, "Good": 2, "n/a": 3}
failures = evaluation.assign(
    context_rank=evaluation["context_quality_auto"].map(RANK),
    answer_rank=evaluation["answer_quality_auto"].map(RANK),
).sort_values(["answer_rank", "context_rank"])
failures = failures[(failures["answer_rank"] < 2) | (failures["context_rank"] < 2)]
failures.drop(columns=["context_rank", "answer_rank"]).to_csv(OUT_DIR / "failure_candidates.csv", index=False)
print("Failure candidates:", len(failures))

for _, row in failures.head(5).iterrows():
    print("=" * 80)
    print(f"{row['embedding_model']} + {row['chunking_strategy']} | {row['query']}")
    print(f"context: {row['context_quality_auto']} | answer: {row['answer_quality_auto']}")
    print("ANSWER:", row["answer"])
    print("CHUNKS:\n" + row["retrieved_chunks"])

Failure candidates: 82
small + overlap | Can I change a contract after it has already been used to check a file?
context: Poor | answer: Poor
ANSWER: No, you cannot change a contract after it has already been used to check a file. A contract version has one of three statuses: draft, active, or retired. Once a contract version is used to check a file, it becomes active and cannot be changed. If you want to change a contract, you need to move it from the draft status to the active status, which replaces the current active version.
CHUNKS:
[1] (0.414) contracts.md > What a contract is + Contract versions: A contract is the written agreement about what the files of a dataset must contain. It belongs to exactly one dataset and holds a list of rules, each describing one checkable expectation about one column. A contract is the thing that makes a validation result meaningful: without it, a file is just data, and there is no standard to measure it against. Contracts are written by people who u

### Step 3.3: Improvement attempt

Each test uses the best setup and every query. `baseline` is the "before" result. The others change
one thing: how many chunks we retrieve, or how strict the prompt is.

In [13]:
STRICT_PROMPT = (
    "You answer questions about DataPact, a data quality tool. "
    "Answer in one or two sentences, using only the numbered context. "
    "Do not add facts that are not in the context. "
    "If the context does not contain the answer, reply only with: "
    "I could not find that in the documents."
)

VARIANTS = {
    "baseline": {"k": TOP_K, "system_prompt": SYSTEM_PROMPT},
    "top_k_1": {"k": 1, "system_prompt": SYSTEM_PROMPT},
    "top_k_5": {"k": 5, "system_prompt": SYSTEM_PROMPT},
    "strict_prompt": {"k": TOP_K, "system_prompt": STRICT_PROMPT},
}

variant_rows = []
for name, settings in VARIANTS.items():
    for query in queries:
        row, _ = run_query(indexes[BEST], query, **settings)
        variant_rows.append({"variant": name, **row})

variants = pd.DataFrame(variant_rows)
variants.to_csv(OUT_DIR / "improvement_results.csv", index=False)
summarize(variants, "variant")

,context_hit_rate,context_phrase_recall,answer_phrase_recall,avg_search_ms,avg_generate_s,declined_correctly
variant,,,,,,
baseline,1.00,0.958,0.472,8.425,0.973,1.0
strict_prompt,1.00,0.958,0.431,7.792,0.750,1.0
top_k_1,0.75,0.500,0.250,7.750,0.729,1.0
top_k_5,1.00,0.958,0.458,7.767,0.926,1.0


## Results files

| File | What is in it |
| --- | --- |
| `evaluation_table.csv` | Step 2.2: one row per setup and query, with full chunks and answers |
| `retrieved_chunks.csv` | The same chunks with one chunk per row |
| `index_stats.csv` | Number of chunks, storage size and build time per setup, for the cost question in Step 4.1 |
| `threshold_scores.csv`, `threshold_summary.csv` | Top score per query and the suggested cutoff for each setup |
| `summary_by_config.csv`, `summary_by_model.csv`, `summary_by_strategy.csv` | Steps 2.3 and 2.4 |
| `scaling_results.csv` | Step 2.5 |
| `failure_candidates.csv` | Step 3.1 |
| `improvement_results.csv` | Step 3.3 |

## Findings

*Written after the full run (AMD Radeon RX 7900 XTX, PyTorch 2.9.1 + ROCm 7.2.1 on Windows,
8 Nov 2026), from the tables above. 9 documents, 40 paragraphs, 14 queries.*

In short: retrieval worked well, and the medium embedding model with fixed-length chunks was the
best setup. It found the right section in the top 3 for all 12 answerable questions. Most of the
problems came from SmolLM2, not from retrieval. It sometimes said it couldn't find an answer that
was right in front of it, and it got a few facts wrong.

**How does embedding size affect retrieval quality?**
Medium found the right section 97% of the time (35 of 36), small 94% (34 of 36) and large 89%
(32 of 36). Bigger was also slower. One search took about 5 ms with small, 9 ms with medium and
16 ms with large. Every miss came from three questions: q04, q08 and q09. These are the medium
and hard questions where two or three sections talk about the same thing.

**How does embedding size affect answer quality?**
Only a little. The answers contained 35% of the expected phrases with small, 41% with medium and
38% with large. The same model writes every answer, so the answer can only be as good as the
chunks it gets. Medium gave it the right chunks most often.

**Did larger embeddings always perform better?**
No. Large was the worst at retrieval and the slowest. It also took the longest to build its index
(0.9 to 1.5 s, against 0.06 s for small) and used almost 3 times the storage. Our questions use
everyday wording and our documents are short, so the extra size did not help.

**Which chunking strategy worked better, and why?**
Fixed length. It found the right section 100% of the time, hybrid 94% and overlap 86%. Overlap
chunks are about 200 words and always mix two sections, so the topic of each chunk is blurrier.
That made it easy for a close but wrong chunk to score higher. Fixed chunks are about 100 words,
so each one is mostly about one thing. Hybrid was close behind, and it missed on q08, where three
sections discuss the same decision.

**How did chunking affect retrieval relevance and the final answers?**
Overlap was worst at retrieval but best at answers (44% of expected phrases, against 35% for
fixed and 33% for hybrid). Its chunks are twice as long, so when it did find the right section the
model had more text to work with. The cost is time. Overlap answers took about 1.5 s, against
0.9 s for fixed, because the prompt is twice as long.

**How does dataset size affect retrieval quality and noise?**
We kept the right chunks and added 25%, 50% and then 100% of the rest (11, 21 and 40 chunks).
The right section was found every time at every size. But the gap between the best right chunk
and the best wrong chunk shrank from 0.22 to 0.19 to 0.13, and the share of wrong chunks in the
top 3 went up from 50% to 56%. More data means more text that looks almost right. Our knowledge
base is small, so it hasn't caused a miss yet, but a bigger one would.

**Failures we saw (for Part 3)**
* **Refusing a question it could answer:** with the normal prompt the model said "I could not find
  that in the documents" for q07 and q12, even though the right chunk was retrieved first.
* **Answer plus refusal:** for q02 it gave the right steps, then added "I could not find that in
  the documents" at the end.
* **Wrong fact:** for q03 it said the statuses are "active, history, and draft". The document says
  retired, and a retired version "is kept for history", so the model grabbed the wrong word.
* **Too short to be useful:** for q08 it only said "The data is out of date.", which skips the
  signals the document gives for deciding.
* **Wrong chunk:** with overlap chunking, q04 and q08 did not get the right section in the top 3.

**Improvement: a stricter prompt**
Before, with the normal prompt, the model refused 3 answerable questions (q02, q07 and q12).
After switching to the strict prompt, it refused none of them and still refused both out-of-scope
questions. The strict prompt tells the model to answer in one or two sentences from the numbered
context, and to reply only with the refusal line if the answer is not there. That stopped it from
tacking a refusal onto a real answer. The phrase score went down a little (47% to 43%) because
the answers are shorter. The prompt did not fix the wrong facts in q03 and q08, because those come
from the model misreading the chunk, not from the instructions.

Changing top-k was not an improvement. With top 1 the right section was found only 75% of the
time. Top 5 found it every time, the same as top 3, so the extra chunks only added text.

**Similarity threshold**
We don't recommend one. In the best setup, the out-of-scope questions scored higher (up to 0.42)
than some real questions (as low as 0.36), so any cutoff would block real questions. Only three
setups had a gap at all, and it was under 0.01, which is too thin to trust with 2 out-of-scope
queries. The prompt handled refusals well instead: both out-of-scope questions were refused in
all 9 setups.

**Recommendation for the app (A9):** medium embeddings (all-mpnet-base-v2), fixed-length chunks,
top 3, the strict prompt and no similarity threshold.